In [1]:
import numpy as np
import pandas as pd
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split

I0000 00:00:1790237209.955256   11463 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1790237211.447845   11463 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1790237215.007588   11463 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


___
# Load Dataset

In [2]:
df = pd.read_csv("WELFake_Dataset.csv")

In [3]:
df

,Unnamed: 0,title,text,label
0,0,LAW ENFORCEMENT ON HIGH ALERT Following Threat...,No comment is expected from Barack Obama Membe...,1
1,1,NaN,Did they post their votes for Hillary already?,1
2,2,UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MO...,"Now, most of the demonstrators gathered last ...",1
3,3,"Bobby Jindal, raised Hindu, uses story of Chri...",A dozen politically active pastors came here f...,0
4,4,SATAN 2: Russia unvelis an image of its terrif...,"The RS-28 Sarmat missile, dubbed Satan 2, will...",1
...,...,...,...,...
72129,72129,Russians steal research on Trump in hack of U....,WASHINGTON (Reuters) - Hackers believed to be ...,0
72130,72130,WATCH: Giuliani Demands That Democrats Apolog...,"You know, because in fantasyland Republicans n...",1
72131,72131,Migrants Refuse To Leave Train At Refugee Camp...,Migrants Refuse To Leave Train At Refugee Camp...,0
72132,72132,Trump tussle gives unpopular Mexican leader mu...,MEXICO CITY (Reuters) - Donald Trump’s combati...,0


In [4]:
# Drop Null Values
df = df.dropna(subset=["text", "label"])

In [5]:
# Combine Ttle + text 

df["content"] = df["title"] + df["text"]

/tmp/ipykernel_11463/2881946390.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["content"] = df["title"] + df["text"]


In [6]:
df

,Unnamed: 0,title,text,label,content
0,0,LAW ENFORCEMENT ON HIGH ALERT Following Threat...,No comment is expected from Barack Obama Membe...,1,LAW ENFORCEMENT ON HIGH ALERT Following Threat...
1,1,NaN,Did they post their votes for Hillary already?,1,NaN
2,2,UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MO...,"Now, most of the demonstrators gathered last ...",1,UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MO...
3,3,"Bobby Jindal, raised Hindu, uses story of Chri...",A dozen politically active pastors came here f...,0,"Bobby Jindal, raised Hindu, uses story of Chri..."
4,4,SATAN 2: Russia unvelis an image of its terrif...,"The RS-28 Sarmat missile, dubbed Satan 2, will...",1,SATAN 2: Russia unvelis an image of its terrif...
...,...,...,...,...,...
72129,72129,Russians steal research on Trump in hack of U....,WASHINGTON (Reuters) - Hackers believed to be ...,0,Russians steal research on Trump in hack of U....
72130,72130,WATCH: Giuliani Demands That Democrats Apolog...,"You know, because in fantasyland Republicans n...",1,WATCH: Giuliani Demands That Democrats Apolog...
72131,72131,Migrants Refuse To Leave Train At Refugee Camp...,Migrants Refuse To Leave Train At Refugee Camp...,0,Migrants Refuse To Leave Train At Refugee Camp...
72132,72132,Trump tussle gives unpopular Mexican leader mu...,MEXICO CITY (Reuters) - Donald Trump’s combati...,0,Trump tussle gives unpopular Mexican leader mu...


___
# data Preprocessing
___

In [7]:
# Drop unncessary Cols 
df = df.drop(columns=['Unnamed: 0', 'title', 'text'])

In [8]:
# Remove rows where content is missing
df = df.dropna(subset=["content"])

# Make sure every remaining value is a string
df["content"] = df["content"].astype(str)

In [9]:
print(df["content"].isna().sum())
print(df["content"].map(type).value_counts())

0
content
<class 'str'>    71537
Name: count, dtype: int64


In [10]:
df

,label,content
0,1,LAW ENFORCEMENT ON HIGH ALERT Following Threat...
2,1,UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MO...
3,0,"Bobby Jindal, raised Hindu, uses story of Chri..."
4,1,SATAN 2: Russia unvelis an image of its terrif...
5,1,About Time! Christian Group Sues Amazon and SP...
...,...,...
72129,0,Russians steal research on Trump in hack of U....
72130,1,WATCH: Giuliani Demands That Democrats Apolog...
72131,0,Migrants Refuse To Leave Train At Refugee Camp...
72132,0,Trump tussle gives unpopular Mexican leader mu...


___
# Tokenizer
___

In [11]:
# tokenize the Text 

tokenizer = Tokenizer(num_words=20000)
# Learn vocabulary from the text
# fit_on_texts() expects a list of strings
tokenizer.fit_on_texts(df["content"]) # learns Vocablary from the content 

In [12]:
total_words = len(tokenizer.word_index) + 1  # 1 as 1 index is reserved for paddng

In [13]:
total_words

362179

In [14]:
tokenizer.word_index

{'the': 1,
 'to': 2,
 'of': 3,
 'and': 4,
 'a': 5,
 'in': 6,
 'that': 7,
 'is': 8,
 'for': 9,
 'on': 10,
 'it': 11,
 'he': 12,
 'with': 13,
 's': 14,
 'was': 15,
 'as': 16,
 'said': 17,
 'by': 18,
 'trump': 19,
 '”': 20,
 'his': 21,
 'be': 22,
 'have': 23,
 'has': 24,
 'not': 25,
 'at': 26,
 'are': 27,
 'from': 28,
 'this': 29,
 'an': 30,
 'who': 31,
 'they': 32,
 'but': 33,
 'i': 34,
 'we': 35,
 'you': 36,
 'about': 37,
 'will': 38,
 'their': 39,
 'would': 40,
 'had': 41,
 'or': 42,
 'been': 43,
 'more': 44,
 'president': 45,
 'were': 46,
 'people': 47,
 'one': 48,
 'her': 49,
 'she': 50,
 'which': 51,
 'all': 52,
 'new': 53,
 'u': 54,
 'if': 55,
 'what': 56,
 'out': 57,
 'after': 58,
 'mr': 59,
 'when': 60,
 'up': 61,
 'clinton': 62,
 'its': 63,
 'there': 64,
 'so': 65,
 'also': 66,
 'no': 67,
 'state': 68,
 'can': 69,
 'over': 70,
 'our': 71,
 'than': 72,
 'like': 73,
 'other': 74,
 '—': 75,
 'just': 76,
 'some': 77,
 'into': 78,
 'him': 79,
 'them': 80,
 'do': 81,
 't': 82,
 'could

___
# Input Sequence
___

``` 
We are not dong the N-gram sequence like in GRU as its not Predictiing Words

In [15]:
sequences = tokenizer.texts_to_sequences(df["content"])

In [16]:
sequences

[[145,
  725,
  10,
  301,
  4233,
  525,
  1368,
  95,
  2572,
  4,
  3191,
  10,
  540,
  10010,
  4,
  1147,
  173,
  67,
  507,
  8,
  570,
  28,
  531,
  87,
  273,
  3,
  1,
  42,
  4,
  10010,
  3750,
  169,
  9,
  1,
  16230,
  4,
  5205,
  3,
  107,
  47,
  4,
  2572,
  32,
  3395,
  389,
  10,
  5,
  1120,
  238,
  261,
  391,
  2,
  735,
  1,
  7730,
  4,
  1466,
  107,
  47,
  4,
  2572,
  2,
  1313,
  5,
  681,
  37,
  1,
  1233,
  3,
  222,
  47,
  6,
  178,
  48,
  3,
  1,
  847,
  4914,
  8,
  169,
  11976,
  50,
  24,
  5,
  1120,
  3091,
  238,
  3458,
  28,
  575,
  169,
  11976,
  14,
  847,
  7473,
  1184,
  1120,
  238,
  5,
  12490,
  3,
  49,
  179,
  1001,
  26,
  540,
  4153,
  963,
  365,
  715,
  7,
  50,
  15,
  3310,
  404,
  2,
  357,
  88,
  2596,
  35,
  481,
  2,
  6387,
  1,
  8303,
  3,
  107,
  777,
  8,
  5,
  12490,
  179,
  1120,
  357,
  6368,
  1120,
  238,
  5957,
  26,
  288,
  3264,
  963,
  365,
  1096,
  1812,
  85,
  130,
  1,
  238,
  98

In [17]:
max_sequence_len = 600

___
# Pad Sequence
___

In [18]:
input_padded = pad_sequences(
    sequences, maxlen = max_sequence_len, padding = "pre"
)

input_padded

array([[    5,    68,     3, ...,   340,   746,   105],
       [    0,     0,     0, ...,     6,     5, 13158],
       [    1,  3703,   761, ...,   274,   174,    20],
       ...,
       [    0,     0,     0, ...,  7610,   417,   686],
       [    0,     0,     0, ...,   484,   696,    20],
       [    0,     0,     0, ...,    22, 15834,   766]],
      shape=(71537, 600), dtype=int32)

___
# X and Y split 
___


In [19]:
X = input_padded
y = df["label"].values

___
# Train Test Split 
___

In [20]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

___
# Model Training
___

In [21]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, GRU, Dropout, Dense


model = Sequential()
model.add(Embedding(total_words, 64, input_length=max_sequence_len))
model.add(Bidirectional(GRU(100)))
model.add(Dropout(0.3))
model.add(Dense(1, activation='sigmoid'))  # binary: fake/real


/home/aizen/miniconda3/envs/gensim-env/lib/python3.12/site-packages/keras/src/layers/core/embedding.py:123: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(
E0000 00:00:1790237278.135884   11463 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [22]:
model.compile(
    optimizer = "adam",
    loss = "binary_crossentropy",
    metrics = ["accuracy"]
)

In [23]:
from tensorflow.keras.callbacks import EarlyStopping

early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

In [ ]:
# Train Model 

history = model.fit(
    X_train,
    y_train,
    validation_data = (X_test, y_test),
    epochs = 50,
    callbacks = [early_stopping],
    verbose = 1
)

___
# Prediction
___

In [ ]:
def predict_news(text):
    sequence = tokenizer.texts_to_sequences([text])
    padded = pad_sequences(sequence, maxlen=max_sequence_len, padding='pre')
    prediction = model.predict(padded, verbose=0)[0][0]
    label = "REAL" if prediction >= 0.5 else "FAKE"
    print(f"Prediction : {label}")
    print(f"Confidence : {prediction:.4f}")

# Test
predict_news("NASA confirms water found on Mars surface in new study")
predict_news("SHOCKING: Government putting chemicals in water to control minds")